In [159]:
import pandas as pd 
# import pycaret.classification as pc 
import pycaret.regression as pr 
import matplotlib.pyplot as plt 
import seaborn as sns 
from scipy.stats import pearsonr
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from IPython.display import display

In [160]:
feat_df = pd.read_csv('data/bookings_listings_merged.csv')
feat_df.head()

,Unnamed: 0.1,Unnamed: 0,listing_id,name,description,neighborhood_overview,host_id,host_since,host_response_time,host_response_rate,...,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,instant_bookable,calculated_host_listings_count,reviews_per_month,booking_ratio
0,0,0,2595.0,Skylit Midtown Castle Sanctuary,"Beautiful, spacious skylit studio in the heart...",Centrally located in the heart of Manhattan ju...,2845,9/9/08,within a day,90%,...,4.73,4.63,4.77,4.80,4.81,4.40,f,3,0.27,0.000000
1,1,1,6848.0,Only 2 stops to Manhattan studio,Comfortable studio apartment with super comfor...,NaN,15991,5/6/09,within a few hours,100%,...,4.59,4.85,4.85,4.80,4.69,4.58,f,1,1.04,0.493151
2,2,2,6872.0,Uptown Sanctuary w/ Private Bath (Month to Month),This charming distancing-friendly month-to-mon...,This sweet Harlem sanctuary is a 10-20 minute ...,16104,5/7/09,a few days or more,30%,...,5.00,5.00,5.00,5.00,5.00,5.00,f,2,0.03,0.772603
3,3,3,6990.0,UES Beautiful Blue Room,Beautiful peaceful healthy home,"Location: Five minutes to Central Park, Museum...",16800,5/12/09,within an hour,100%,...,4.83,4.95,4.96,4.95,4.85,4.85,f,1,1.37,0.221918
4,4,4,7064.0,"Amazing location! Wburg. Large, bright & tranquil","Large, private loft-like room in a spacious 2-...","- One stop from the East Village, Lower East S...",17297,5/15/09,NaN,NaN,...,5.00,4.91,5.00,5.00,5.00,5.00,f,2,0.08,1.000000


In [161]:
feat_df.columns

Index(['Unnamed: 0.1', 'Unnamed: 0', 'listing_id', 'name', 'description',
       'neighborhood_overview', 'host_id', 'host_since', 'host_response_time',
       'host_response_rate', 'host_acceptance_rate', 'host_is_superhost',
       'host_listings_count', 'host_total_listings_count',
       'host_identity_verified', 'neighbourhood_cleansed',
       'neighbourhood_group_cleansed', 'property_type', 'room_type',
       'accommodates', 'bathrooms', 'bathrooms_text', 'bedrooms', 'beds',
       'amenities', 'price', 'minimum_nights', 'maximum_nights',
       'minimum_minimum_nights', 'maximum_minimum_nights',
       'minimum_maximum_nights', 'maximum_maximum_nights',
       'minimum_nights_avg_ntm', 'maximum_nights_avg_ntm', 'has_availability',
       'availability_365', 'calendar_last_scraped', 'number_of_reviews',
       'number_of_reviews_ltm', 'number_of_reviews_l30d', 'first_review',
       'last_review', 'review_scores_rating', 'review_scores_accuracy',
       'review_scores_cleanline

In [162]:
feat_df = feat_df.drop(columns=['Unnamed: 0', 'Unnamed: 0.1', 'name', 'listing_id', 'description','neighbourhood_group_cleansed', 'neighborhood_overview','host_id', 'host_since', 'host_response_time', 'bathrooms_text','first_review', 'last_review', 'availability_365', 'host_listings_count', 'number_of_reviews_ltm', 'number_of_reviews_l30d', 'review_scores_accuracy', 'calendar_last_scraped' ])


In [163]:
feat_df = feat_df.loc[(feat_df['booking_ratio'] > 0.1) & (feat_df['booking_ratio'] <= 0.7)]
feat_df.head()

,host_response_rate,host_acceptance_rate,host_is_superhost,host_total_listings_count,host_identity_verified,neighbourhood_cleansed,property_type,room_type,accommodates,bathrooms,...,review_scores_rating,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,instant_bookable,calculated_host_listings_count,reviews_per_month,booking_ratio
1,100%,100%,t,1.0,t,Williamsburg,Entire rental unit,Entire home/apt,3,1.0,...,4.58,4.85,4.85,4.80,4.69,4.58,f,1,1.04,0.493151
3,100%,100%,t,6.0,t,East Harlem,Private room in rental unit,Private room,1,1.0,...,4.88,4.95,4.96,4.95,4.85,4.85,f,1,1.37,0.221918
5,100%,100%,t,2.0,t,Fort Greene,Private room in guest suite,Private room,2,1.0,...,4.89,4.89,4.96,4.93,4.95,4.82,t,2,2.16,0.410959
6,100%,100%,t,4.0,t,Williamsburg,Entire place,Entire home/apt,2,1.0,...,4.91,4.67,4.89,4.78,5.00,4.89,f,1,0.07,0.290411
7,100%,NaN,f,3.0,t,Bedford-Stuyvesant,Entire loft,Entire home/apt,5,1.0,...,4.77,4.74,4.88,4.88,4.67,4.76,f,2,1.03,0.400000


In [164]:
feat_df['price'] = feat_df['price'].replace({'\\$': '', ',': ''}, regex=True)
feat_df['price'] = pd.to_numeric(feat_df['price'], errors='coerce')

feat_df['host_response_rate'] = feat_df['host_response_rate'].replace ({'\\%': '', ',': ''}, regex=True)
feat_df['host_response_rate'] = pd.to_numeric(feat_df['host_response_rate'], errors="coerce")

feat_df['host_acceptance_rate'] = feat_df['host_response_rate'].replace ({'\\%': '', ',': ''}, regex=True)
feat_df['host_acceptance_rate'] = pd.to_numeric(feat_df['host_acceptance_rate'], errors="coerce")

In [177]:
feat_df.head()

,host_response_rate,host_acceptance_rate,host_is_superhost,host_total_listings_count,host_identity_verified,neighbourhood_cleansed,property_type,room_type,accommodates,bathrooms,...,review_scores_rating,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,instant_bookable,calculated_host_listings_count,reviews_per_month,booking_ratio
1,100.0,100.0,t,1.0,t,Williamsburg,Entire rental unit,Entire home/apt,3,1.0,...,4.58,4.85,4.85,4.80,4.69,4.58,f,1,1.04,0.493151
3,100.0,100.0,t,6.0,t,East Harlem,Private room in rental unit,Private room,1,1.0,...,4.88,4.95,4.96,4.95,4.85,4.85,f,1,1.37,0.221918
5,100.0,100.0,t,2.0,t,Fort Greene,Private room in guest suite,Private room,2,1.0,...,4.89,4.89,4.96,4.93,4.95,4.82,t,2,2.16,0.410959
6,100.0,100.0,t,4.0,t,Williamsburg,Entire place,Entire home/apt,2,1.0,...,4.91,4.67,4.89,4.78,5.00,4.89,f,1,0.07,0.290411
7,100.0,100.0,f,3.0,t,Bedford-Stuyvesant,Entire loft,Entire home/apt,5,1.0,...,4.77,4.74,4.88,4.88,4.67,4.76,f,2,1.03,0.400000


In [178]:
print(feat_df.dtypes)

host_response_rate                float64
host_acceptance_rate              float64
host_is_superhost                  object
host_total_listings_count         float64
host_identity_verified             object
neighbourhood_cleansed             object
property_type                      object
room_type                          object
accommodates                        int64
bathrooms                         float64
bedrooms                          float64
beds                              float64
amenities                          object
price                             float64
minimum_nights                      int64
maximum_nights                      int64
minimum_minimum_nights            float64
maximum_minimum_nights            float64
minimum_maximum_nights            float64
maximum_maximum_nights            float64
minimum_nights_avg_ntm            float64
maximum_nights_avg_ntm            float64
has_availability                   object
number_of_reviews                 

In [ ]:
demo = pr.setup(data = feat_df, target = 'booking_ratio', session_id=42) 

In [ ]:
best_model = pr.compare_models()

In [ ]:
pr.plot_model(best_model, plot='feature')

In [ ]:
pr.plot_model(best_model,plot='residuals')

## Define x train and y train for ridge regression

In [179]:
# Define features (X) and target (y)
X = feat_df.drop(columns=["booking_ratio"])  # Replace "target" with the actual column name
y = feat_df["booking_ratio"]  # Target variable

## Handle Missing Values for Each Column

In [181]:
# Numerical columns

num_cols = [
    "host_response_rate", "host_acceptance_rate", "bathrooms", "bedrooms", 
    "beds", "price", "review_scores_rating", "review_scores_cleanliness", 
    "review_scores_checkin", "review_scores_communication", 
    "review_scores_location", "review_scores_value", "reviews_per_month"
]

for col in num_cols:
    X[col].fillna(X[col].median(), inplace=True)


# Binary columns
binary_cols = ["host_is_superhost", "host_identity_verified", "has_availability", "instant_bookable"]

for col in binary_cols:
    X[col] = X[col].replace({"t": 1, "f": 0})  # Convert "t"/"f" to 1/0
    X[col] = X[col].fillna(X[col].mode()[0] if not X[col].mode().empty else 0)  # Fill missing with mode or default to 0

In [170]:
print(X.isnull().sum().sum())  # Should print 0 if all NaNs are handled


0


In [171]:
print(X.isnull().sum()[X.isnull().sum() > 0])  # Show only columns with NaNs


Series([], dtype: int64)


In [172]:
print(X.isnull().sum()[X.isnull().sum() > 0])  # Show only columns with NaNs


Series([], dtype: int64)


##  Encode Categorical Variables

In [182]:
# # Binary categorical columns (Convert "t"/"f" to 0/1):
# binary_cols = ["host_is_superhost", "host_identity_verified", "has_availability", "instant_bookable"]

# for col in binary_cols:
#     X[col] = X[col].map({"t": 1, "f": 0})

# #Multi-class categorical columns
X = pd.get_dummies(X, columns=["neighbourhood_cleansed", "property_type", "room_type"], drop_first=True)

#The amenities column contains lists of amenities. Convert it into a count of amenities.
# X["amenities_count"] = X["amenities"].apply(lambda x: len(eval(x)) if isinstance(x, str) else 0)

# Drop the original amenities column
X.drop(columns=["amenities"], inplace=True)

## Scale Features for Ridge Regression

In [183]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

##  Split Data into Training and Testing Sets

In [184]:
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

## Hyperparameter tuning for Ridge Regression

### GridSearch

In [189]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV
import numpy as np

# Define hyperparameter grid for Ridge
alpha_values = np.logspace(-3, 3, 50)  # Testing alpha from 0.001 to 1000

param_grid = {"alpha": alpha_values}

# Initialize Ridge model
ridge = Ridge()

# Perform Grid Search with Cross-Validation
grid_search_ridge = GridSearchCV(
    ridge, param_grid, cv=5, scoring="neg_mean_absolute_error", n_jobs=-1
)
grid_search_ridge.fit(X_train, y_train)

# Best hyperparameter and performance
print("Best alpha for Ridge - Grid Search:", grid_search_ridge.best_params_["alpha"])
print("Best MAE for Ridge - Grid Search:", -grid_search_ridge.best_score_)

Best alpha for Ridge - Grid Search: 8.286427728546842
Best MAE for Ridge - Grid Search: 0.15697976809034464


### RandomizedSearch

In [190]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform

# Define search space (uniform distribution for alpha)
param_dist = {"alpha": uniform(0.001, 1000)}

# Randomized Search with 50 iterations
random_search_ridge = RandomizedSearchCV(
    Ridge(), param_distributions=param_dist, 
    n_iter=50, cv=5, scoring="neg_mean_absolute_error", 
    random_state=42, n_jobs=-1
)
random_search_ridge.fit(X_train, y_train)

# Best alpha
print("Best alpha for Ridge - Random Search:", random_search_ridge.best_params_["alpha"])
print("Best MAE for Ridge - Random Search:", -random_search_ridge.best_score_)

Best alpha for Ridge - Random Search: 20.585494295802448
Best MAE for Ridge - Random Search: 0.15701017459235525


In [187]:
best_alpha = grid_search_ridge.best_params_["alpha"]
ridge_best = Ridge(alpha=best_alpha)
ridge_best.fit(X_train, y_train)

# Predict on test set
y_pred = ridge_best.predict(X_test)

# Evaluate final performance
from sklearn.metrics import mean_absolute_error
print("Final Ridge Test MAE:", mean_absolute_error(y_test, y_pred))

Final Ridge Test MAE: 0.1617930796163497


In [191]:
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import train_test_split

# Example with RidgeCV
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

ridge = RidgeCV(alphas=[0.1, 1, 10, 100, 1000], store_cv_values=True)
ridge.fit(X_train, y_train)

# Best alpha found
print(ridge.alpha_)


10.0
